# Closing-auction volume — feature construction template (E0–E5)

Loads the raw data, cleans it, and builds the model-ready feature table for the E0–E5 specifications from the midterm presentation.

**To use:** set `DATA_ROOT` in the first code cell, then run all cells (about 1 minute).

**Forecast being supported:** for stock *i* on trading day *t*, predict the NYSE closing-auction share volume `A(i,t)` using information available by the close of *t − 1*.

**Output** (in `outputs/`)

- `model_features/year=YYYY/…parquet` — one row per stock-day with the target and every feature
- `feature_dictionary.csv` — the formula, raw source and timing of every feature

| Step | Features |
|---|---|
| E0 | Historical auction-volume reference (20-row median) and the target |
| E1 | Index rebalancing events |
| E2 | Monthly options expiration (plus optional stock-level options exposure) |
| E3 | Month-end, quarter-end, year-end, early closes, start of month |
| E4 | Scheduled FOMC meeting dates |
| E5 | Lagged volatility and trading-volume regimes, with event interactions |

Checks on this pipeline (look-ahead test, cross-checks, comparison with the midterm results) are kept separately in `02_checks_optional.ipynb`.

## 0. Settings

In [1]:
from pathlib import Path

DATA_ROOT = None                  # path to the repo's `blackrock_data` folder; None = find it automatically
REFERENCE_DIR = Path("reference_inputs")
OUTPUT_DIR = Path("outputs")

MODEL_START = "2015-01-01"        # auction data starts in 2015; earlier CRSP rows are used as history only
MIN_AUCTION_HISTORY = 10          # rows need this many auction prints in the previous 20 rows to be model-ready
USE_OPTIONS = True                # add stock-level options exposure (needs stock_daily/options_daily)
USE_SP500_STOCK_LEVEL = True      # add S&P 500 membership and add/delete features

In [2]:
from datetime import date, timedelta
import numpy as np
import pandas as pd
import duckdb                      # pip install duckdb
from dateutil.easter import easter

pd.set_option("display.max_columns", 80); pd.set_option("display.width", 220)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

if DATA_ROOT is None:              # look for blackrock_data next to or above this notebook
    here = Path.cwd().resolve()
    options_ = [p / "blackrock_data" for p in [here, *here.parents]] + \
               [p / "blackrock_auction_vol" / "blackrock_data" for p in [here, *here.parents]]
    DATA_ROOT = next((p for p in options_ if p.exists()), None)
assert DATA_ROOT is not None and Path(DATA_ROOT).exists(), "Set DATA_ROOT to the blackrock_data folder."
DATA_ROOT = Path(DATA_ROOT)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
USE_OPTIONS = USE_OPTIONS and (DATA_ROOT / "stock_daily" / "options_daily").exists()
print("Data folder:", DATA_ROOT)

Data folder: /Users/kunj/Desktop/blackrock_auction_vol/blackrock_data


## 1. Load the raw data

| Raw table | Source | Key |
|---|---|---|
| **CRSP daily** | WRDS `crsp.dsf_v2` + `crsp.stksecurityinfohist`. NYSE-primary, US-incorporated ordinary common shares, regular-way, actively trading. No size filter. | `date`, `permno` |
| **Auction daily** | WRDS TAQ trades. The official NYSE closing-auction print: exchange `N`, correction `00`, sale condition containing `6`. Linked to `permno` with a date-specific TAQ–CRSP bridge. | `date`, `permno` |
| **Options daily** (optional) | WRDS OptionMetrics, aggregated to one row per stock-day (`11_options_daily.ipynb`). | `date`, `permno` |

In [3]:
def read_folder(folder, columns, where="TRUE"):
    """Read all parquet files under a folder into one DataFrame."""
    source = f"read_parquet('{Path(folder) / '**' / '*.parquet'}', union_by_name=true, hive_partitioning=false)"
    out = duckdb.connect().execute(f"SELECT {', '.join(columns)} FROM {source} WHERE {where}").df()
    out["date"] = pd.to_datetime(out["date"]).astype("datetime64[ns]")
    out["permno"] = out["permno"].astype("int64")
    return out

# CRSP: re-apply the universe filter on load
UNIVERSE = ("primary_exchange = 'N' AND security_type = 'EQTY' AND security_subtype = 'COM' AND share_type = 'NS' "
            "AND us_incorporated = 'Y' AND issuer_type = 'CORP' AND conditional_type = 'RW' AND trading_status = 'A'")
crsp_raw = read_folder(DATA_ROOT / "stock_daily" / "crsp_daily",
                       ["date", "permno", "ticker", "close", "daily_volume", "daily_price_volume", "return_total", "market_cap_usd"], UNIVERSE)
auction_raw = read_folder(DATA_ROOT / "stock_daily" / "auction_daily", ["date", "permno", "auction_volume", "target_found"])

OPTION_ASOF_PRIOR_CLOSE = ["opt_call_oi", "opt_put_oi", "opt_total_oi", "oi_exp_today", "oi_exp_7d"]   # open interest on row t = close of t-1
OPTION_SAME_DAY = ["has_options", "opt_total_vol", "opt_to_stock_vol", "iv_atm_30d", "gex_dollar"]        # known only after the close of t
options_raw = read_folder(DATA_ROOT / "stock_daily" / "options_daily",
                          ["date", "permno"] + OPTION_ASOF_PRIOR_CLOSE + OPTION_SAME_DAY) if USE_OPTIONS else None

for name, df in [("CRSP daily", crsp_raw), ("Auction daily", auction_raw), ("Options daily", options_raw)]:
    if df is not None:
        print(f"{name:14s}: {len(df):>10,} rows  ({df.date.min():%Y-%m-%d} to {df.date.max():%Y-%m-%d})")

CRSP daily    :  4,927,078 rows  (2011-01-03 to 2025-12-31)
Auction daily :  3,594,398 rows  (2015-01-02 to 2025-12-31)
Options daily :  3,488,919 rows  (2015-01-02 to 2025-08-29)


## 2. Clean

- One row per stock-day; the CRSP rows are the spine.
- An auction observation counts only when `target_found` is true and `auction_volume` is present. **A missing print stays missing, never zero.**
- 2019-08-12 is excluded from the target: the NYSE auction feed was incomplete that day.

In [4]:
EXCLUDED_DATES = [pd.Timestamp("2019-08-12")]

def clean_inputs(crsp, auction):
    assert not crsp.duplicated(["date", "permno"]).any() and not auction.duplicated(["date", "permno"]).any()
    a = auction.copy()
    a["observed_target"] = a["target_found"].eq(True) & a["auction_volume"].notna()
    a.loc[~a["observed_target"], "auction_volume"] = np.nan
    panel = crsp.merge(a[["date", "permno", "auction_volume", "observed_target"]], on=["date", "permno"], how="left")
    panel["observed_target"] = panel["observed_target"].eq(True)
    panel["eligible_target"] = panel["observed_target"] & ~panel["date"].isin(EXCLUDED_DATES)
    return panel.sort_values(["permno", "date"]).reset_index(drop=True)

preview = clean_inputs(crsp_raw, auction_raw)
(preview[preview.date >= MODEL_START].assign(year=lambda d: d.date.dt.year)
 .groupby("year").agg(stock_days=("permno", "size"), stocks=("permno", "nunique"), eligible_targets=("eligible_target", "sum")))

,stock_days,stocks,eligible_targets
year,,,
2015,341909,1426,337566
2016,329950,1386,325638
2017,321370,1348,318040
2018,317656,1329,313807
2019,316550,1316,308240
2020,312434,1320,303168
2021,336769,1477,321564
2022,353277,1469,328634
2023,331648,1397,318679


## 3. Trading calendar

Built from NYSE rules, so flags like "last trading day of the month" also work for future dates.

- **Holidays**: New Year's Day, MLK Day, Presidents' Day, Good Friday, Memorial Day, Juneteenth (from 2022), Independence Day, Labor Day, Thanksgiving, Christmas. Sunday holidays are observed Monday; Saturday holidays are observed Friday (except New Year's Day).
- **Unscheduled closures**: 2012-10-29, 2012-10-30, 2018-12-05, 2025-01-09.
- **Early closes (1:00 p.m.)**: day after Thanksgiving; July 3 and December 24 when they are trading days. The closing auction still runs on these days, at 1:00 p.m.

In [5]:
SPECIAL_CLOSURES = [date(2012, 10, 29), date(2012, 10, 30), date(2018, 12, 5), date(2025, 1, 9)]

def nth_weekday(year, month, weekday, n):
    """n-th given weekday (Mon=0) of a month; n=-1 is the last one."""
    if n > 0:
        d = date(year, month, 1)
        return d + timedelta(days=(weekday - d.weekday()) % 7, weeks=n - 1)
    d = date(year, month + 1, 1) - timedelta(days=1) if month < 12 else date(year, 12, 31)
    return d - timedelta(days=(d.weekday() - weekday) % 7)

def observed(d, saturday_moves_back=True):
    if d.weekday() == 6: return d + timedelta(days=1)
    if d.weekday() == 5: return d - timedelta(days=1) if saturday_moves_back else None
    return d

def nyse_holidays(year):
    h = [observed(date(year, 1, 1), saturday_moves_back=False), nth_weekday(year, 1, 0, 3), nth_weekday(year, 2, 0, 3),
         easter(year) - timedelta(days=2), nth_weekday(year, 5, 0, -1), observed(date(year, 7, 4)),
         nth_weekday(year, 9, 0, 1), nth_weekday(year, 11, 3, 4), observed(date(year, 12, 25))]
    if year >= 2022: h.append(observed(date(year, 6, 19)))
    return {d for d in h if d is not None} | {d for d in SPECIAL_CLOSURES if d.year == year}

def trading_days(start_year, end_year):
    holidays = set().union(*[nyse_holidays(y) for y in range(start_year, end_year + 1)])
    return pd.DatetimeIndex([d for d in pd.bdate_range(f"{start_year}-01-01", f"{end_year}-12-31") if d.date() not in holidays])

data_days = pd.DatetimeIndex(sorted(crsp_raw["date"].unique()))
TRADING_DAYS = trading_days(data_days[0].year, data_days[-1].year + 1)      # one extra year for forward-looking flags
assert data_days.isin(TRADING_DAYS).all(), "A date in the data is not a trading day under the calendar rules."

## 4. Reference lists

Dates that cannot be derived by rule are kept in small CSV files, each with its public source.

| File | Content | Source |
|---|---|---|
| `fomc_meetings.csv` | FOMC meetings 2015–2026 | Federal Reserve FOMC calendar pages |
| `index_event_dates.csv` | MSCI index-review dates (and Russell / S&P schedule dates) | MSCI, FTSE Russell, S&P Dow Jones Indices |
| `sp500_constituent_changes.csv` | S&P 500 additions and deletions: announcement date and trade date | Team index-rebalance dataset |
| `sp500_membership_spells.csv` | S&P 500 membership intervals by `permno` | Team index-rebalance dataset |

In [6]:
fomc = pd.read_csv(REFERENCE_DIR / "fomc_meetings.csv", parse_dates=["meeting_start_date", "announcement_date"])
index_events = pd.read_csv(REFERENCE_DIR / "index_event_dates.csv", parse_dates=["date"])
sp_changes = pd.read_csv(REFERENCE_DIR / "sp500_constituent_changes.csv", parse_dates=["announcement_date", "trade_date"])
sp_spells = pd.read_csv(REFERENCE_DIR / "sp500_membership_spells.csv", parse_dates=["member_from", "member_to"])

# Scheduled meetings only: unscheduled meetings cannot be known in advance
fomc_dates = fomc.loc[(fomc.meeting_type == "scheduled") & (fomc.status != "cancelled"), "announcement_date"]
msci_dates = index_events.loc[index_events.event_type == "msci_review_effective", "date"]

## 5. Calendar and event flags (E1, E2, E3, E4)

All known in advance from the calendar and published schedules.

| Flag | Rule | Step |
|---|---|---|
| `sp_quarterly_rebal` | Third Friday of Mar/Jun/Sep/Dec. Also the quarterly options expiration, so one flag covers both. | E1 |
| `russell_recon` | Fourth Friday of June. | E1 |
| `msci_review` | Effective date of each MSCI index review (reference list). | E1 |
| `monthly_opex_ex_quarterly` | Third Friday of the other eight months. | E2 |
| `is_last_trading_day_month / quarter / year`, `is_first_trading_day_month`, `is_early_close` | From the trading calendar. | E3 |
| `fomc_announcement_day` | Statement day (second day) of a scheduled FOMC meeting. | E4 |

If a rule date is a holiday, the previous trading day is used.

In [7]:
def previous_trading_day(d, days):
    d = pd.Timestamp(d)
    return d if d in days else days[days < d][-1]

def build_calendar(days):
    cal = pd.DataFrame({"date": days})
    cal["year"] = cal.date.dt.year
    month, quarter = cal.date.dt.to_period("M"), cal.date.dt.to_period("Q")
    years = range(days[0].year, days[-1].year + 1)
    third_friday = {(y, m): previous_trading_day(nth_weekday(y, m, 4, 3), days) for y in years for m in range(1, 13)}

    # E1 - index rebalancing events
    cal["sp_quarterly_rebal"] = cal.date.isin({d for (y, m), d in third_friday.items() if m in (3, 6, 9, 12)}).astype("int8")
    cal["russell_recon"] = cal.date.isin({previous_trading_day(nth_weekday(y, 6, 4, 4), days) for y in years}).astype("int8")
    cal["msci_review"] = cal.date.isin(set(msci_dates)).astype("int8")
    cal["any_index_event"] = cal[["sp_quarterly_rebal", "russell_recon", "msci_review"]].max(axis=1).astype("int8")
    # E2 - monthly options expiration, excluding quarterly
    cal["monthly_opex_ex_quarterly"] = (cal.date.isin(set(third_friday.values())).astype("int8") - cal["sp_quarterly_rebal"]).astype("int8")
    # E3 - institutional calendar
    cal["is_last_trading_day_month"] = (month != month.shift(-1)).astype("int8")
    cal["is_last_trading_day_quarter"] = (quarter != quarter.shift(-1)).astype("int8")
    cal["is_last_trading_day_year"] = (cal.year != cal.year.shift(-1)).astype("int8")
    cal["is_first_trading_day_month"] = (month != month.shift(1)).astype("int8")
    early = {previous_trading_day(nth_weekday(y, 11, 3, 4) + timedelta(days=1), days) for y in years}
    early |= {pd.Timestamp(y, 7, 3) for y in years} | {pd.Timestamp(y, 12, 24) for y in years}
    cal["is_early_close"] = cal.date.isin(early).astype("int8")
    # E4 - FOMC
    cal["fomc_announcement_day"] = cal.date.isin(set(fomc_dates)).astype("int8")
    return cal.iloc[:-1].reset_index(drop=True)

DATE_FLAGS = ["sp_quarterly_rebal", "russell_recon", "msci_review", "any_index_event", "monthly_opex_ex_quarterly",
              "is_last_trading_day_month", "is_last_trading_day_quarter", "is_last_trading_day_year",
              "is_first_trading_day_month", "is_early_close", "fomc_announcement_day"]
calendar = build_calendar(TRADING_DAYS)
calendar[calendar.date.isin(data_days) & (calendar.date >= MODEL_START)].groupby("year")[DATE_FLAGS].sum()      # event days per year

,sp_quarterly_rebal,russell_recon,msci_review,any_index_event,monthly_opex_ex_quarterly,is_last_trading_day_month,is_last_trading_day_quarter,is_last_trading_day_year,is_first_trading_day_month,is_early_close,fomc_announcement_day
year,,,,,,,,,,,
2015,4,1,4,9,8,12,4,1,12,2,8
2016,4,1,4,9,8,12,4,1,12,1,8
2017,4,1,4,9,8,12,4,1,12,2,8
2018,4,1,4,9,8,12,4,1,12,3,8
2019,4,1,4,9,8,12,4,1,12,3,8
2020,4,1,4,9,8,12,4,1,12,2,7
2021,4,1,4,9,8,12,4,1,12,1,8
2022,4,1,4,9,8,12,4,1,12,1,8
2023,4,1,4,9,8,12,4,1,12,2,8


## 6. E0 — historical reference and the target

For each stock, in date order:

- `median20_auction_volume` $= M_{20}(i,t)$: median auction volume over the stock's **previous 20 rows** (*t − 20 … t − 1*), skipping missing prints.
- `n_obs_median20`: number of prints behind that median.
- Target: `y_log_ratio` $= \log(1 + A(i,t)) - \log(1 + M_{20}(i,t))$.

In [8]:
def prior_window(panel, column, window, how, min_periods=1):
    """Statistic of `column` over the stock's previous `window` rows (t-window ... t-1; never row t)."""
    shifted = panel.groupby("permno", sort=False)[column].shift(1)
    rolled = shifted.groupby(panel["permno"], sort=False).rolling(window, min_periods=min_periods)
    return getattr(rolled, how)().reset_index(level=0, drop=True)

def add_e0(panel):
    panel["median20_auction_volume"] = prior_window(panel, "auction_volume", 20, "median")
    panel["n_obs_median20"] = prior_window(panel, "auction_volume", 20, "count").fillna(0).astype("int16")
    panel["log_median20_auction_volume"] = np.log1p(panel["median20_auction_volume"])
    panel["y_log_ratio"] = np.log1p(panel["auction_volume"]) - panel["log_median20_auction_volume"]
    return panel

E0_FEATURES = ["median20_auction_volume", "log_median20_auction_volume", "n_obs_median20"]

## 7. E1 — S&P 500 membership and announced changes (optional)

- `sp500_member_lag1`: in the S&P 500 on the previous trading day.
- `sp500_member_x_sp_rebal`: member × quarterly rebalance day.
- `sp500_add_pending`, `sp500_del_pending`: from the trading day after the announcement through the trade date.
- `sp500_add_trade_day`, `sp500_del_trade_day`: the trade date itself (the close at which index funds trade).
- `td_to_sp500_change`: trading days until the trade date while a change is pending.

Changes with a missing or estimated announcement date are not used.

In [9]:
SP_FEATURES = ["sp500_member_lag1", "sp500_member_x_sp_rebal", "sp500_add_pending", "sp500_del_pending",
               "sp500_add_trade_day", "sp500_del_trade_day", "td_to_sp500_change"]

def add_sp500(panel, days):
    day_number = pd.Series(np.arange(len(days)), index=days)
    previous_day = pd.Series(days[:-1], index=days[1:])
    keys = panel[["permno", "date"]].assign(prev_date=panel["date"].map(previous_day)).merge(sp_spells, on="permno")
    member = keys[(keys.prev_date >= keys.member_from) & (keys.prev_date <= keys.member_to)][["permno", "date"]].drop_duplicates()
    panel = panel.merge(member.assign(sp500_member_lag1=1), on=["permno", "date"], how="left")
    panel["sp500_member_lag1"] = panel["sp500_member_lag1"].fillna(0).astype("int8")
    panel["sp500_member_x_sp_rebal"] = (panel["sp500_member_lag1"] * panel["sp_quarterly_rebal"]).astype("int8")

    usable = sp_changes[(sp_changes.announcement_date_estimated == 0) & sp_changes.announcement_date.notna()
                        & (sp_changes.announcement_date < sp_changes.trade_date) & sp_changes.trade_date.isin(days)]
    rows = [(ev.permno, d, "add" if ev.event == "add" else "del", int(day_number[ev.trade_date] - day_number[d]))
            for ev in usable.itertuples() for d in days[(days > ev.announcement_date) & (days <= ev.trade_date)]]
    pending = pd.DataFrame(rows, columns=["permno", "date", "kind", "td"])
    for kind in ("add", "del"):
        p = pending[pending.kind == kind].groupby(["permno", "date"], as_index=False)["td"].min()
        p[f"sp500_{kind}_pending"], p[f"sp500_{kind}_trade_day"] = 1, (p["td"] == 0).astype(int)
        panel = panel.merge(p.rename(columns={"td": f"td_{kind}"}), on=["permno", "date"], how="left")
        for c in (f"sp500_{kind}_pending", f"sp500_{kind}_trade_day"):
            panel[c] = panel[c].fillna(0).astype("int8")
    panel["td_to_sp500_change"] = panel[["td_add", "td_del"]].min(axis=1)
    return panel.drop(columns=["td_add", "td_del"])

## 8. E2 extension — stock-level options exposure (optional)

The E2 calendar flag says *whether* options expire; these say *how much* is expiring for each stock.

**Timing.** OptionMetrics open interest on row *t* is the open interest at the close of *t − 1*, so it can be used for day *t*.
Volume, implied volatility and gamma on row *t* are only known after the close of *t*, so they are lagged one row.

- `opt_oi_exp_today_share`, `opt_oi_exp_7d_share`: open interest expiring today / within 7 days ÷ total open interest.
- `opt_log_total_oi`: log(1 + total open interest). `opt_pc_oi_ratio`: put ÷ call open interest.
- `opt_has_options_lag1`, `opt_to_stock_vol_lag1`, `opt_iv_atm_30d_lag1`: previous row's values.
- `opt_abs_gex_to_dollar_volume_lag1`: previous row's |dealer gamma exposure| ÷ 20-row average dollar volume.

In [10]:
OPT_FEATURES = ["opt_oi_exp_today_share", "opt_oi_exp_7d_share", "opt_log_total_oi", "opt_pc_oi_ratio",
                "opt_has_options_lag1", "opt_to_stock_vol_lag1", "opt_iv_atm_30d_lag1", "opt_abs_gex_to_dollar_volume_lag1"]

def add_options(panel, options):
    panel = panel.merge(options, on=["date", "permno"], how="left")
    total_oi = panel["opt_total_oi"].where(panel["opt_total_oi"] > 0)
    panel["opt_oi_exp_today_share"] = panel["oi_exp_today"] / total_oi
    panel["opt_oi_exp_7d_share"] = panel["oi_exp_7d"] / total_oi
    panel["opt_log_total_oi"] = np.log1p(panel["opt_total_oi"])
    panel["opt_pc_oi_ratio"] = panel["opt_put_oi"] / panel["opt_call_oi"].where(panel["opt_call_oi"] > 0)
    lagged = panel.groupby("permno", sort=False)[OPTION_SAME_DAY].shift(1)
    panel["opt_has_options_lag1"] = lagged["has_options"]
    panel["opt_to_stock_vol_lag1"] = lagged["opt_to_stock_vol"]
    panel["opt_iv_atm_30d_lag1"] = lagged["iv_atm_30d"]
    avg20_dollar_volume = prior_window(panel, "daily_price_volume", 20, "mean")
    panel["opt_abs_gex_to_dollar_volume_lag1"] = lagged["gex_dollar"].abs() / avg20_dollar_volume.where(avg20_dollar_volume > 0)
    return panel.drop(columns=OPTION_ASOF_PRIOR_CLOSE + OPTION_SAME_DAY)

## 9. E5 — lagged volatility and volume regimes, with event interactions

**Stock** (own previous rows)

- `volatility20`: standard deviation of daily returns over the previous 20 rows.
- `abnormal_volume_lag1`: previous row's volume ÷ average volume over the previous 20 rows.
- `size_pct_lag1`: percentile rank (0–1) of the previous row's market cap across stocks that day.

**Market** (from the CRSP cross-section, lagged)

- `market_volatility20`: standard deviation over the previous 20 days of the value-weighted market return.
- `market_abnormal_volume_lag1`: previous day's total volume ÷ its average over the previous 20 days.

**Regimes**

- `high_vol_regime`: `market_volatility20` above its median over the trailing 252 days.
- `high_volume_regime`: `market_abnormal_volume_lag1` above 1.

**Interactions**: `index_event_x_high_vol`, `opex_x_high_vol`, `month_end_x_high_vol`, `russell_recon_x_small_cap` (= `russell_recon × (1 − size_pct_lag1)`).

In [11]:
E5_FEATURES = ["volatility20", "abnormal_volume_lag1", "size_pct_lag1", "market_volatility20", "market_abnormal_volume_lag1",
               "high_vol_regime", "high_volume_regime", "index_event_x_high_vol", "opex_x_high_vol", "month_end_x_high_vol",
               "russell_recon_x_small_cap"]

def add_e5_stock(panel):
    panel["volatility20"] = prior_window(panel, "return_total", 20, "std", min_periods=2)
    avg20_volume = prior_window(panel, "daily_volume", 20, "mean")
    panel["abnormal_volume_lag1"] = panel.groupby("permno", sort=False)["daily_volume"].shift(1) / avg20_volume.where(avg20_volume > 0)
    lag1_market_cap = panel.groupby("permno", sort=False)["market_cap_usd"].shift(1)
    panel["size_pct_lag1"] = lag1_market_cap.groupby(panel["date"]).rank(pct=True)
    return panel

def market_series(crsp):
    """Daily value-weighted market return and total volume across the universe."""
    ok = crsp["return_total"].notna() & crsp["market_cap_usd"].notna()
    return pd.DataFrame({
        "market_return_vw": (crsp["return_total"] * crsp["market_cap_usd"]).where(ok).groupby(crsp["date"]).sum()
                            / crsp["market_cap_usd"].where(ok).groupby(crsp["date"]).sum(),
        "market_total_volume": crsp.groupby("date")["daily_volume"].sum(min_count=1),
    }).sort_index()

def market_regimes(cal, market):
    m = market.reindex(cal["date"])
    out = pd.DataFrame({"date": cal["date"].values})
    out["market_volatility20"] = m["market_return_vw"].shift(1).rolling(20, min_periods=2).std().values
    lag1_volume = m["market_total_volume"].shift(1)
    out["market_abnormal_volume_lag1"] = (lag1_volume / lag1_volume.rolling(20, min_periods=1).mean()).values
    trailing_median = out["market_volatility20"].rolling(252, min_periods=126).median()
    out["high_vol_regime"] = (out["market_volatility20"] > trailing_median).astype(float).where(trailing_median.notna())
    out["high_volume_regime"] = (out["market_abnormal_volume_lag1"] > 1).astype(float).where(out["market_abnormal_volume_lag1"].notna())
    return out

def add_interactions(panel):
    panel["index_event_x_high_vol"] = panel["any_index_event"] * panel["high_vol_regime"]
    panel["opex_x_high_vol"] = panel["monthly_opex_ex_quarterly"] * panel["high_vol_regime"]
    panel["month_end_x_high_vol"] = panel["is_last_trading_day_month"] * panel["high_vol_regime"]
    panel["russell_recon_x_small_cap"] = panel["russell_recon"] * (1 - panel["size_pct_lag1"])
    return panel

## 10. Build the feature table

In [12]:
def build_features(crsp, auction, options, cal):
    panel = clean_inputs(crsp, auction)
    panel = add_e0(panel)
    panel = add_e5_stock(panel)
    if options is not None:
        panel = add_options(panel, options)
    panel = panel.merge(cal[["date"] + DATE_FLAGS].merge(market_regimes(cal, market_series(crsp)), on="date"), on="date", how="left")
    if USE_SP500_STOCK_LEVEL:
        panel = add_sp500(panel, pd.DatetimeIndex(cal["date"]))
    panel = add_interactions(panel)
    panel["model_ready"] = panel["eligible_target"] & (panel["n_obs_median20"] >= MIN_AUCTION_HISTORY)
    return panel

ID_COLUMNS = ["date", "permno", "ticker"]
TARGET_COLUMNS = ["auction_volume", "eligible_target", "y_log_ratio", "model_ready"]
FEATURE_COLUMNS = (E0_FEATURES + DATE_FLAGS + (SP_FEATURES if USE_SP500_STOCK_LEVEL else [])
                   + (OPT_FEATURES if USE_OPTIONS else []) + E5_FEATURES)

full = build_features(crsp_raw, auction_raw, options_raw, calendar)
features = full.loc[full.date >= MODEL_START, ID_COLUMNS + TARGET_COLUMNS + FEATURE_COLUMNS].reset_index(drop=True)
print(f"{len(features):,} stock-days, {len(FEATURE_COLUMNS)} features, {int(features.model_ready.sum()):,} model-ready rows")
features.tail(3).T

3,594,404 stock-days, 40 features, 3,466,653 model-ready rows


,3594401,3594402,3594403
date,2025-12-29 00:00:00,2025-12-30 00:00:00,2025-12-31 00:00:00
permno,93426,93426,93426
ticker,VPG,VPG,VPG
auction_volume,10829,6102,11548
eligible_target,True,True,True
y_log_ratio,-0.0143,-0.5878,0.0500
model_ready,True,True,True
median20_auction_volume,"10,985.0000","10,985.0000","10,985.0000"
log_median20_auction_volume,9.3044,9.3044,9.3044
n_obs_median20,20,20,20


## 11. Feature dictionary and save

`known_by` says when each feature is available: **calendar** = fixed by a published schedule, known well in advance; **prior close** = uses data through *t − 1*;
**pre-open** = open interest as of the close of *t − 1* (published by the OCC before the open of *t*).

In [13]:
DICTIONARY = [  # feature, step, formula, raw source, known_by
 ("median20_auction_volume", "E0", "Median auction_volume over the stock's previous 20 rows, skipping missing prints", "TAQ auction prints", "prior close"),
 ("log_median20_auction_volume", "E0", "log(1 + median20_auction_volume)", "TAQ auction prints", "prior close"),
 ("n_obs_median20", "E0", "Number of auction prints among the stock's previous 20 rows", "TAQ auction prints", "prior close"),
 ("sp_quarterly_rebal", "E1", "1 on the third Friday of Mar/Jun/Sep/Dec (also quarterly options expiration)", "Calendar rule", "calendar"),
 ("russell_recon", "E1", "1 on the fourth Friday of June", "Calendar rule", "calendar"),
 ("msci_review", "E1", "1 on the effective date of an MSCI index review", "index_event_dates.csv", "calendar"),
 ("any_index_event", "E1", "max(sp_quarterly_rebal, russell_recon, msci_review)", "Derived", "calendar"),
 ("monthly_opex_ex_quarterly", "E2", "1 on the third Friday of Jan, Feb, Apr, May, Jul, Aug, Oct, Nov", "Calendar rule", "calendar"),
 ("is_last_trading_day_month", "E3", "1 on the last trading day of the month", "Trading calendar", "calendar"),
 ("is_last_trading_day_quarter", "E3", "1 on the last trading day of the quarter", "Trading calendar", "calendar"),
 ("is_last_trading_day_year", "E3", "1 on the last trading day of the year", "Trading calendar", "calendar"),
 ("is_first_trading_day_month", "E3", "1 on the first trading day of the month", "Trading calendar", "calendar"),
 ("is_early_close", "E3", "1 on 1:00 p.m. close days (day after Thanksgiving; July 3 and Dec 24 if trading days)", "Trading calendar", "calendar"),
 ("fomc_announcement_day", "E4", "1 on the statement day of a scheduled FOMC meeting", "fomc_meetings.csv", "calendar"),
 ("sp500_member_lag1", "E1", "1 if in the S&P 500 on the previous trading day", "sp500_membership_spells.csv", "prior close"),
 ("sp500_member_x_sp_rebal", "E1", "sp500_member_lag1 * sp_quarterly_rebal", "Derived", "prior close"),
 ("sp500_add_pending", "E1", "1 from the day after an addition is announced through its trade date", "sp500_constituent_changes.csv", "prior close"),
 ("sp500_del_pending", "E1", "1 from the day after a deletion is announced through its trade date", "sp500_constituent_changes.csv", "prior close"),
 ("sp500_add_trade_day", "E1", "1 on the trade date of an announced addition", "sp500_constituent_changes.csv", "prior close"),
 ("sp500_del_trade_day", "E1", "1 on the trade date of an announced deletion", "sp500_constituent_changes.csv", "prior close"),
 ("td_to_sp500_change", "E1", "Trading days to the trade date while a change is pending", "sp500_constituent_changes.csv", "prior close"),
 ("opt_oi_exp_today_share", "E2+", "Open interest expiring on t / total open interest", "OptionMetrics", "pre-open"),
 ("opt_oi_exp_7d_share", "E2+", "Open interest expiring within 7 calendar days / total open interest", "OptionMetrics", "pre-open"),
 ("opt_log_total_oi", "E2+", "log(1 + total call and put open interest)", "OptionMetrics", "pre-open"),
 ("opt_pc_oi_ratio", "E2+", "Put open interest / call open interest", "OptionMetrics", "pre-open"),
 ("opt_has_options_lag1", "E2+", "1 if the stock had listed options on its previous row", "OptionMetrics", "prior close"),
 ("opt_to_stock_vol_lag1", "E2+", "Previous row's option contracts traded * 100 / share volume", "OptionMetrics, CRSP", "prior close"),
 ("opt_iv_atm_30d_lag1", "E2+", "Previous row's 30-day at-the-money implied volatility", "OptionMetrics", "prior close"),
 ("opt_abs_gex_to_dollar_volume_lag1", "E2+", "abs(previous row's dealer gamma exposure in dollars) / mean dollar volume over previous 20 rows", "OptionMetrics, CRSP", "prior close"),
 ("volatility20", "E5", "Std of return_total over the stock's previous 20 rows", "CRSP", "prior close"),
 ("abnormal_volume_lag1", "E5", "Previous row's daily_volume / mean daily_volume over previous 20 rows", "CRSP", "prior close"),
 ("size_pct_lag1", "E5", "Percentile rank across stocks of the previous row's market cap", "CRSP", "prior close"),
 ("market_volatility20", "E5", "Std over previous 20 days of the value-weighted market return", "CRSP", "prior close"),
 ("market_abnormal_volume_lag1", "E5", "Previous day's total market volume / its mean over previous 20 days", "CRSP", "prior close"),
 ("high_vol_regime", "E5", "1 if market_volatility20 > its median over the trailing 252 days", "Derived", "prior close"),
 ("high_volume_regime", "E5", "1 if market_abnormal_volume_lag1 > 1", "Derived", "prior close"),
 ("index_event_x_high_vol", "E5", "any_index_event * high_vol_regime", "Derived", "prior close"),
 ("opex_x_high_vol", "E5", "monthly_opex_ex_quarterly * high_vol_regime", "Derived", "prior close"),
 ("month_end_x_high_vol", "E5", "is_last_trading_day_month * high_vol_regime", "Derived", "prior close"),
 ("russell_recon_x_small_cap", "E5", "russell_recon * (1 - size_pct_lag1)", "Derived", "prior close"),
]
dictionary = pd.DataFrame(DICTIONARY, columns=["feature", "step", "formula", "raw_source", "known_by"])
dictionary = dictionary[dictionary.feature.isin(FEATURE_COLUMNS)].reset_index(drop=True)
assert set(dictionary.feature) == set(FEATURE_COLUMNS)
dictionary.to_csv(OUTPUT_DIR / "feature_dictionary.csv", index=False)

for year, chunk in features.groupby(features.date.dt.year):
    folder = OUTPUT_DIR / "model_features" / f"year={year}"
    folder.mkdir(parents=True, exist_ok=True)
    chunk.to_parquet(folder / f"model_features_{year}.parquet", index=False, compression="zstd")
print(f"Saved {len(features):,} rows to {OUTPUT_DIR / 'model_features'}")
dictionary

Saved 3,594,404 rows to outputs/model_features


,feature,step,formula,raw_source,known_by
0,median20_auction_volume,E0,Median auction_volume over the stock's previou...,TAQ auction prints,prior close
1,log_median20_auction_volume,E0,log(1 + median20_auction_volume),TAQ auction prints,prior close
2,n_obs_median20,E0,Number of auction prints among the stock's pre...,TAQ auction prints,prior close
3,sp_quarterly_rebal,E1,1 on the third Friday of Mar/Jun/Sep/Dec (also...,Calendar rule,calendar
4,russell_recon,E1,1 on the fourth Friday of June,Calendar rule,calendar
5,msci_review,E1,1 on the effective date of an MSCI index review,index_event_dates.csv,calendar
6,any_index_event,E1,"max(sp_quarterly_rebal, russell_recon, msci_re...",Derived,calendar
7,monthly_opex_ex_quarterly,E2,"1 on the third Friday of Jan, Feb, Apr, May, J...",Calendar rule,calendar
8,is_last_trading_day_month,E3,1 on the last trading day of the month,Trading calendar,calendar
9,is_last_trading_day_quarter,E3,1 on the last trading day of the quarter,Trading calendar,calendar
